# Cross-Entropy Method Planning Trials

Standalone notebook adaptation of the `rl-agents` Cross-Entropy Method planner for `highway-env`.

Source inspiration:
- `rl_agents/agents/cross_entropy_method/cem.py`
- `scripts/configs/HighwayEnv/agents/CEMAgent/cem.json`

Unlike DQN and PPO, CEM here is a sampling-based planner, not a trained policy. It uses the environment itself as an oracle model by deep-copying the current state and rolling out candidate continuous action sequences.

In [ ]:
from __future__ import annotations

import copy
import json
import sys
from pathlib import Path

import gymnasium as gym
import highway_env  # noqa: F401
import matplotlib
import numpy as np
import pandas as pd
import torch

try:
    from IPython.display import Image, display
except ImportError:
    class Image:
        def __init__(self, filename=None, data=None, format=None):
            self.filename = filename
            self.data = data
            self.format = format

        def __repr__(self):
            return f"Image(filename={self.filename!r})"

    def display(*items):
        for item in items:
            print(item)

matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import imageio.v2 as imageio
except ImportError:
    imageio = None


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root()
RL_AGENTS_ROOT = PROJECT_ROOT / "rl-agents"
if str(RL_AGENTS_ROOT) not in sys.path:
    sys.path.insert(0, str(RL_AGENTS_ROOT))

from rl_agents.agents.common.factory import safe_deepcopy_env as factory_safe_deepcopy_env

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "planning"
RESULTS_DIR = PROJECT_ROOT / "artifacts" / "planning" / "cem"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("rl-agents root:", RL_AGENTS_ROOT)
print("Notebook dir:", NOTEBOOK_DIR)
print("Results dir:", RESULTS_DIR)
print(f"Torch version: {torch.__version__}")


## Config

The `ContinuousAction` space is required because Leurent's CEM agent is designed for continuous action spaces.

In [2]:
RUN_NAME = "cem_highway_baseline"

ENV_CONFIG = {
    "observation": {
        "type": "Kinematics",
        "vehicles_count": 5,
        "features": ["presence", "x", "y", "vx", "vy"],
        "absolute": False,
    },
    "action": {
        "type": "ContinuousAction",
        "longitudinal": True,
        "lateral": True,
        "dynamical": False,
    },
    "lanes_count": 3,
    "vehicles_count": 20,
    "duration": 40,
}

PLANNER_CONFIG = {
    "gamma": 0.99,
    "horizon": 6,
    "iterations": 5,
    "candidates": 256,
    "top_candidates": 32,
    "init_std": 0.7,
    "min_std": 0.05,
}

EVAL_CONFIG = {
    "episodes": 5,
    "seed": 42,
    "show_progress": True,
    "episode_progress_every_steps": 5,
    "show_planner_iteration_progress": True,
    "capture_render": True,
    "render_episode": 1,
    "render_max_frames": 250,
    "render_fps": 12,
}

print(json.dumps({
    "run_name": RUN_NAME,
    "env": ENV_CONFIG,
    "planner": PLANNER_CONFIG,
    "eval": EVAL_CONFIG,
}, indent=2))


{
  "run_name": "cem_highway_baseline",
  "env": {
    "observation": {
      "type": "Kinematics",
      "vehicles_count": 5,
      "features": [
        "presence",
        "x",
        "y",
        "vx",
        "vy"
      ],
      "absolute": false
    },
    "action": {
      "type": "ContinuousAction",
      "longitudinal": true,
      "lateral": true,
      "dynamical": false
    },
    "lanes_count": 3,
    "vehicles_count": 20,
    "duration": 40
  },
  "planner": {
    "gamma": 0.99,
    "horizon": 6,
    "iterations": 5,
    "candidates": 256,
    "top_candidates": 32,
    "init_std": 0.7,
    "min_std": 0.05
  },
  "eval": {
    "episodes": 5,
    "seed": 42
  }
}


## Planner And Evaluation Helpers

In [ ]:
def make_env(config: dict, render_mode=None):
    return gym.make("highway-v0", render_mode=render_mode, config=config)


def safe_deepcopy_env(env):
    # Strip viewer/video handles before copying so CEM rollouts do not clone live pygame state.
    return factory_safe_deepcopy_env(env)


def capture_frame(env):
    frame = env.render()
    if frame is None:
        return None
    return np.asarray(frame).copy()


def save_gif(frames, path: Path, fps: int = 12):
    if not frames:
        return None
    if imageio is None:
        raise ImportError("imageio is required to save GIF renders. Install it or set capture_render=False.")
    path.parent.mkdir(parents=True, exist_ok=True)
    imageio.mimsave(path, frames, fps=max(1, int(fps)))
    return path


def compute_same_lane_ttc(env, ttc_cap: float = 10.0) -> float:
    vehicle = getattr(env.unwrapped, "vehicle", None)
    road = getattr(env.unwrapped, "road", None)
    if vehicle is None or road is None:
        return float(ttc_cap)

    lane_index = getattr(vehicle, "lane_index", None)
    if lane_index is None:
        return float(ttc_cap)

    front_vehicle, _ = road.neighbour_vehicles(vehicle, lane_index)
    if front_vehicle is None:
        return float(ttc_cap)

    lane = road.network.get_lane(lane_index)
    ego_s, _ = lane.local_coordinates(vehicle.position)
    front_s, _ = lane.local_coordinates(front_vehicle.position)
    clearance = max(
        0.0,
        float(front_s - ego_s)
        - 0.5 * float(getattr(vehicle, "LENGTH", 0.0) + getattr(front_vehicle, "LENGTH", 0.0)),
    )

    ego_speed = float(vehicle.speed * np.cos(getattr(vehicle, "heading", 0.0)))
    front_speed = float(front_vehicle.speed * np.cos(getattr(front_vehicle, "heading", 0.0)))
    closing_speed = ego_speed - front_speed
    if closing_speed <= 1e-6:
        return float(ttc_cap)

    ttc = 0.0 if clearance <= 0.0 else clearance / closing_speed
    return float(np.clip(ttc, 0.0, ttc_cap))


def update_overtake_tracker(env, seen_ahead_ids: set[int], overtaken_ids: set[int]) -> None:
    vehicle = getattr(env.unwrapped, "vehicle", None)
    road = getattr(env.unwrapped, "road", None)
    if vehicle is None or road is None:
        return

    ego_x = float(vehicle.position[0])
    for other in getattr(road, "vehicles", []):
        if other is vehicle:
            continue
        other_id = id(other)
        dx = float(other.position[0] - ego_x)
        if dx > 0.0:
            seen_ahead_ids.add(other_id)
        elif other_id in seen_ahead_ids:
            overtaken_ids.add(other_id)


class CEMPlanner:
    """Standalone adaptation of Leurent's continuous-action CEM planner."""

    def __init__(self, env, config: dict):
        self.env = env
        self.config = {
            "gamma": 1.0,
            "horizon": 10,
            "iterations": 10,
            "candidates": 100,
            "top_candidates": 10,
            "init_std": 1.0,
            "min_std": 0.05,
            **config,
        }
        self.action_size = int(env.action_space.shape[0])
        self.low = torch.as_tensor(env.action_space.low, dtype=torch.float32)
        self.high = torch.as_tensor(env.action_space.high, dtype=torch.float32)

    def plan(self, observation=None, *, progress_prefix: str | None = None, show_progress: bool = False):
        horizon = int(self.config["horizon"])
        candidates_count = int(self.config["candidates"])
        elite_count = int(self.config["top_candidates"])
        gamma = float(self.config["gamma"])
        min_std = float(self.config["min_std"])
        iterations = int(self.config["iterations"])

        mean = torch.zeros(horizon, self.action_size, dtype=torch.float32)
        std = torch.full(
            (horizon, self.action_size),
            float(self.config["init_std"]),
            dtype=torch.float32,
        )

        for iteration_idx in range(iterations):
            distribution = torch.distributions.Normal(mean, std)
            actions = distribution.sample((candidates_count,))
            actions = torch.maximum(torch.minimum(actions, self.high.view(1, 1, -1)), self.low.view(1, 1, -1))

            candidate_envs = [safe_deepcopy_env(self.env) for _ in range(candidates_count)]
            returns = torch.zeros(candidates_count, dtype=torch.float32)
            active = np.ones(candidates_count, dtype=bool)

            for t in range(horizon):
                action_batch = actions[:, t, :].cpu().numpy().astype(np.float32)
                for c, candidate in enumerate(candidate_envs):
                    if not active[c]:
                        continue
                    _, reward, terminated, truncated, _ = candidate.step(action_batch[c])
                    returns[c] += (gamma ** t) * float(reward)
                    if terminated or truncated:
                        active[c] = False
                if not active.any():
                    break

            _, topk = returns.topk(elite_count, largest=True, sorted=False)
            best_actions = actions[topk]
            elite_returns = returns[topk]
            mean = best_actions.mean(dim=0)
            std = best_actions.std(dim=0, unbiased=False).clamp_min(min_std)
            if show_progress:
                prefix = progress_prefix or "[planner]"
                print(
                    f"{prefix} iter={iteration_idx + 1}/{iterations} "
                    f"best_return={returns.max().item():.2f} "
                    f"elite_mean={elite_returns.mean().item():.2f} "
                    f"std_mean={std.mean().item():.3f}",
                    flush=True,
                )

        return mean[0].cpu().numpy().astype(np.float32)

    def act(self, observation=None, **plan_kwargs):
        return self.plan(observation, **plan_kwargs)


def evaluate_planner(
    env_config: dict,
    planner_config: dict,
    episodes: int,
    seed: int,
    show_progress: bool = False,
    episode_progress_every_steps: int = 5,
    show_planner_iteration_progress: bool = False,
    capture_render: bool = False,
    render_episode: int = 1,
    render_max_frames: int = 250,
    render_fps: int = 12,
    render_dir: Path | None = None,
):
    env = make_env(env_config, render_mode="rgb_array" if capture_render else None)
    summaries = []
    try:
        for episode_idx in range(int(episodes)):
            obs, _ = env.reset(seed=seed + episode_idx)
            should_capture = bool(capture_render and (episode_idx + 1 == int(render_episode)))
            episode_frames = []
            if should_capture:
                first_frame = capture_frame(env)
                if first_frame is not None:
                    episode_frames.append(first_frame)
            planner = CEMPlanner(env, planner_config)
            terminated = False
            truncated = False
            total_reward = 0.0
            speed_trace = []
            ttc_trace = []
            seen_ahead_ids = set()
            overtaken_ids = set()
            final_info = {}
            step_count = 0
            if show_progress:
                print(f"[eval] starting episode {episode_idx + 1}/{episodes}", flush=True)

            while not (terminated or truncated):
                update_overtake_tracker(env, seen_ahead_ids, overtaken_ids)
                speed_trace.append(float(getattr(env.unwrapped.vehicle, "speed", 0.0)))
                ttc_trace.append(compute_same_lane_ttc(env))

                step_count += 1
                planner_prefix = f"[eval][episode {episode_idx + 1}/{episodes}][step {step_count}]"
                action = planner.act(
                    obs,
                    progress_prefix=planner_prefix,
                    show_progress=show_progress and show_planner_iteration_progress,
                )
                obs, reward, terminated, truncated, info = env.step(action)
                if should_capture and len(episode_frames) < int(render_max_frames):
                    frame = capture_frame(env)
                    if frame is not None:
                        episode_frames.append(frame)
                total_reward += float(reward)
                final_info = info
                if show_progress and step_count % max(1, int(episode_progress_every_steps)) == 0:
                    print(
                        f"[eval] episode={episode_idx + 1}/{episodes} step={step_count} "
                        f"reward_so_far={total_reward:.2f} overtakes={len(overtaken_ids)} "
                        f"avg_speed={np.mean(speed_trace):.2f}",
                        flush=True,
                    )

            collision = bool(final_info.get("crashed", getattr(env.unwrapped.vehicle, "crashed", False)))
            summaries.append(
                {
                    "episode": int(episode_idx + 1),
                    "reward": float(total_reward),
                    "collision": bool(collision),
                    "avg_speed": float(np.mean(speed_trace)) if speed_trace else 0.0,
                    "overtakes": int(len(overtaken_ids)),
                    "avg_ttc": float(np.mean(ttc_trace)) if ttc_trace else 10.0,
                    "min_ttc": float(np.min(ttc_trace)) if ttc_trace else 10.0,
                    "steps": int(step_count),
                    "render_path": None,
                }
            )
            if should_capture and episode_frames:
                render_base = render_dir or Path.cwd() / "renders"
                render_path = render_base / f"episode_{episode_idx + 1:02d}.gif"
                saved_path = save_gif(episode_frames, render_path, fps=render_fps)
                summaries[-1]["render_path"] = str(saved_path) if saved_path is not None else None
            print(
                f"[eval] episode={episode_idx + 1}/{episodes} reward={total_reward:.2f} "
                f"collision={collision} overtakes={len(overtaken_ids)} avg_speed={np.mean(speed_trace):.2f}",
                flush=True,
            )
    finally:
        env.close()
    return summaries


def plot_evaluation_metrics(summaries, save_path: Path) -> None:
    if not summaries:
        return

    episodes = np.arange(1, len(summaries) + 1)
    rewards = np.array([float(item["reward"]) for item in summaries], dtype=float)
    steps = np.array([float(item["steps"]) for item in summaries], dtype=float)
    avg_speed = np.array([float(item["avg_speed"]) for item in summaries], dtype=float)
    overtakes = np.array([float(item["overtakes"]) for item in summaries], dtype=float)
    avg_ttc = np.array([float(item["avg_ttc"]) for item in summaries], dtype=float)
    min_ttc = np.array([float(item["min_ttc"]) for item in summaries], dtype=float)
    collisions = np.array([float(bool(item["collision"])) for item in summaries], dtype=float)
    running_collision_rate = 100.0 * np.cumsum(collisions) / np.arange(1, len(collisions) + 1)

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))

    axes[0, 0].plot(episodes, rewards, marker="o", color="tab:gray")
    axes[0, 0].set_title("Episode Reward")
    axes[0, 0].set_xlabel("Episode")
    axes[0, 0].set_ylabel("Reward")
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(episodes, avg_speed, marker="o", color="tab:green")
    axes[0, 1].set_title("Average Speed")
    axes[0, 1].set_xlabel("Episode")
    axes[0, 1].set_ylabel("m/s")
    axes[0, 1].grid(alpha=0.3)

    axes[0, 2].plot(episodes, running_collision_rate, marker="o", color="crimson")
    axes[0, 2].set_title("Running Collision Rate")
    axes[0, 2].set_xlabel("Episode")
    axes[0, 2].set_ylabel("%")
    axes[0, 2].set_ylim(0.0, 100.0)
    axes[0, 2].grid(alpha=0.3)

    axes[1, 0].bar(episodes, overtakes, color="tab:orange")
    axes[1, 0].set_title("Overtakes")
    axes[1, 0].set_xlabel("Episode")
    axes[1, 0].set_ylabel("Count")
    axes[1, 0].grid(axis="y", alpha=0.3)

    axes[1, 1].plot(episodes, steps, marker="o", color="tab:brown")
    axes[1, 1].set_title("Episode Length")
    axes[1, 1].set_xlabel("Episode")
    axes[1, 1].set_ylabel("Steps")
    axes[1, 1].grid(alpha=0.3)

    axes[1, 2].plot(episodes, min_ttc, marker="o", label="Min TTC", color="tab:blue")
    axes[1, 2].plot(episodes, avg_ttc, marker="o", label="Avg TTC", color="tab:purple")
    axes[1, 2].set_title("Time To Collision")
    axes[1, 2].set_xlabel("Episode")
    axes[1, 2].set_ylabel("Seconds")
    axes[1, 2].grid(alpha=0.3)
    axes[1, 2].legend()

    fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def build_metric_summary(eval_df: pd.DataFrame) -> pd.DataFrame:
    summary_rows = []
    metric_specs = [
        ("reward", "Reward", 1.0),
        ("steps", "Episode Length (steps)", 1.0),
        ("avg_speed", "Average Speed (m/s)", 1.0),
        ("collision", "Collision Rate (%)", 100.0),
        ("overtakes", "Overtakes", 1.0),
        ("avg_ttc", "Average TTC (s)", 1.0),
        ("min_ttc", "Minimum TTC (s)", 1.0),
    ]
    for column, label, scale in metric_specs:
        values = pd.to_numeric(eval_df[column], errors="coerce").fillna(0.0).astype(float) * scale
        summary_rows.append(
            {
                "metric": label,
                "mean": float(values.mean()),
                "std": float(values.std(ddof=0)),
                "min": float(values.min()),
                "max": float(values.max()),
            }
        )
    return pd.DataFrame(summary_rows)


def plot_metric_summary(eval_df: pd.DataFrame, save_path: Path, title: str) -> pd.DataFrame:
    metric_specs = [
        ("reward", "Reward", "tab:gray", 1.0),
        ("steps", "Episode Length (steps)", "tab:brown", 1.0),
        ("avg_speed", "Average Speed (m/s)", "tab:green", 1.0),
        ("collision", "Collision Rate (%)", "crimson", 100.0),
        ("overtakes", "Overtakes", "tab:orange", 1.0),
        ("avg_ttc", "Average TTC (s)", "tab:purple", 1.0),
        ("min_ttc", "Minimum TTC (s)", "tab:blue", 1.0),
    ]
    ncols = 3
    nrows = int(np.ceil(len(metric_specs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows))
    axes = np.atleast_1d(axes).ravel()
    episode_label = f"{len(eval_df)} eps"
    for ax, (column, label, color, scale) in zip(axes, metric_specs):
        values = pd.to_numeric(eval_df[column], errors="coerce").fillna(0.0).astype(float) * scale
        mean = float(values.mean())
        std = float(values.std(ddof=0))
        ax.bar([0], [mean], yerr=[std], capsize=8, color=color, alpha=0.85)
        ax.scatter(np.zeros(len(values)), values, color=color, alpha=0.18, s=18)
        ax.set_xticks([0], [episode_label])
        ax.set_title(f"{label}\nmean={mean:.2f}, std={std:.2f}")
        ax.grid(axis="y", alpha=0.3)
    for ax in axes[len(metric_specs):]:
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return build_metric_summary(eval_df)


## Evaluate The Planner

In [ ]:
run_dir = RESULTS_DIR / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)

summaries = evaluate_planner(
    env_config=ENV_CONFIG,
    planner_config=PLANNER_CONFIG,
    episodes=EVAL_CONFIG["episodes"],
    seed=EVAL_CONFIG["seed"],
    show_progress=EVAL_CONFIG.get("show_progress", False),
    episode_progress_every_steps=EVAL_CONFIG.get("episode_progress_every_steps", 5),
    show_planner_iteration_progress=EVAL_CONFIG.get("show_planner_iteration_progress", False),
    capture_render=EVAL_CONFIG.get("capture_render", False),
    render_episode=EVAL_CONFIG.get("render_episode", 1),
    render_max_frames=EVAL_CONFIG.get("render_max_frames", 250),
    render_fps=EVAL_CONFIG.get("render_fps", 12),
    render_dir=run_dir / "renders",
)

eval_df = pd.DataFrame(summaries)
metrics_path = run_dir / "evaluation_metrics.json"
metrics_path.write_text(eval_df.to_json(orient="records", indent=2), encoding="utf-8")

detailed_plot_path = run_dir / "evaluation_metrics.png"
plot_evaluation_metrics(summaries, detailed_plot_path)

metric_summary_path = run_dir / "evaluation_metric_summary.json"
summary_plot_path = run_dir / "evaluation_summary.png"
metric_summary_df = plot_metric_summary(
    eval_df,
    summary_plot_path,
    f"CEM Evaluation Summary ({len(eval_df)} episodes)",
)
metric_summary_path.write_text(metric_summary_df.to_json(orient="records", indent=2), encoding="utf-8")

summary = {
    "run_name": RUN_NAME,
    "episodes": int(len(eval_df)),
    "mean_reward": float(pd.to_numeric(eval_df["reward"], errors="coerce").mean()),
    "std_reward": float(pd.to_numeric(eval_df["reward"], errors="coerce").std(ddof=0)),
    "collision_rate_percent": float(100.0 * pd.to_numeric(eval_df["collision"], errors="coerce").mean()),
    "mean_avg_speed": float(pd.to_numeric(eval_df["avg_speed"], errors="coerce").mean()),
    "mean_overtakes": float(pd.to_numeric(eval_df["overtakes"], errors="coerce").mean()),
    "mean_avg_ttc": float(pd.to_numeric(eval_df["avg_ttc"], errors="coerce").mean()),
    "mean_min_ttc": float(pd.to_numeric(eval_df["min_ttc"], errors="coerce").mean()),
    "mean_steps": float(pd.to_numeric(eval_df["steps"], errors="coerce").mean()),
    "evaluation_metrics_path": str(metrics_path),
    "evaluation_plot_path": str(detailed_plot_path),
    "evaluation_metric_summary_path": str(metric_summary_path),
    "evaluation_summary_plot_path": str(summary_plot_path),
    "metric_summary": metric_summary_df.to_dict(orient="records"),
}
summary_path = run_dir / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))
display(Image(filename=detailed_plot_path))
display(Image(filename=summary_plot_path))
display(metric_summary_df.round(3))
display(eval_df[["episode", "steps", "collision", "avg_speed", "overtakes", "avg_ttc", "min_ttc", "reward", "render_path"]])

render_rows = [item for item in summaries if item.get("render_path")]
if render_rows:
    display(Image(filename=render_rows[0]["render_path"]))


: 

## Human Rendering Demo

Run this cell manually to watch 5 live episodes with the current CEM planner settings. This uses `render_mode="human"`, so it opens the simulator viewer instead of saving inline frames.

In [ ]:
human_env = make_env(ENV_CONFIG, render_mode="human")
human_summaries = []

try:
    for episode_idx in range(5):
        obs, _ = human_env.reset(seed=EVAL_CONFIG["seed"] + episode_idx)
        planner = CEMPlanner(human_env, PLANNER_CONFIG)
        terminated = False
        truncated = False
        total_reward = 0.0
        step_count = 0
        final_info = {}

        while not (terminated or truncated):
            action = planner.act(obs)
            obs, reward, terminated, truncated, info = human_env.step(action)
            human_env.render()
            total_reward += float(reward)
            step_count += 1
            final_info = info

        crashed = bool(final_info.get("crashed", getattr(human_env.unwrapped.vehicle, "crashed", False)))
        human_summaries.append(
            {
                "episode": int(episode_idx + 1),
                "steps": int(step_count),
                "reward": float(total_reward),
                "collision": bool(crashed),
            }
        )
        print(
            f"[human] episode={episode_idx + 1}/5 reward={total_reward:.2f} "
            f"steps={step_count} collision={crashed}",
            flush=True,
        )
finally:
    human_env.close()

pd.DataFrame(human_summaries)
